<a href="https://colab.research.google.com/github/Kaue-Melo/Projects/blob/main/Refinamento_Catalogo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
from google.colab import files

print("Selecione o arquivo de catálogo (Excel ou CSV):")
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

# 1. Carregar os dados
if nome_arquivo.endswith('.csv'):
    df = pd.read_csv(nome_arquivo)
else:
    df = pd.read_excel(nome_arquivo)

# 2. Preparar o Dicionário (Coluna: 'Dicionario', Coluna: 'SkuRevisado', Coluna: 'Alerta') -- OBRIGATÓRIO TER AS COLUNAS EM SEU ARQUIVO.
df_ref = df[['Dicionario', 'SkuRevisado', 'Alerta']].dropna(subset=['Dicionario'])

mapa_correcao = {}
for _, row in df_ref.iterrows():
    termo = str(row['Dicionario']).strip()
    mapa_correcao[termo] = {
        'correcao': str(row['SkuRevisado']).strip(),
        'alerta': str(row['Alerta']).strip() if pd.notna(row['Alerta']) and str(row['Alerta']).strip() != "" else None
    }

# ORDENAÇÃO CRÍTICA: Garante que "Pedaco" venha antes de "Ped"
sorted_keys = sorted(mapa_correcao.keys(), key=len, reverse=True)

def limpar_descricao_robusta(texto):
    if not isinstance(texto, str):
        return texto, "OK"

    texto_trabalho = texto.title()
    alertas_encontrados = []

    # 2. Substituição em ordem decrescente de tamanho
    for erro in sorted_keys:
        dados = mapa_correcao[erro]

        # O \b garante que 'Ped' não substitua o começo de 'Pedaco' -- NÃO MUDAR
        # O re.escape garante que caracteres como / ou . NÃO QUEBREM O CÓDIGO
        pattern = rf'\b{re.escape(erro)}\b'

        # Verificamos se o erro existe antes de tentar o re.sub (para coletar o alerta)
        if re.search(pattern, texto_trabalho, flags=re.IGNORECASE):
            texto_trabalho = re.sub(pattern, dados['correcao'], texto_trabalho, flags=re.IGNORECASE)
            if dados['alerta']:
                alertas_encontrados.append(dados['alerta'])

    # 3. Normalização de Unidades (Passo final para garantir estética)
    units = {
        r'(\d+)\s*Kg\b': r'\1kg', r'(\d+)\s*G\b': r'\1g', r'(\d+)\s*Mg\b': r'\1mg',
        r'(\d+)\s*Un\b': r'\1un', r'(\d+)\s*Ml\b': r'\1ml',
        r'\bKg\b': 'kg', r'\bG\b': 'g'
    }
    for p, r in units.items():
        texto_trabalho = re.sub(p, r, texto_trabalho, flags=re.IGNORECASE)

    texto_final = re.sub(r'\s+', ' ', texto_trabalho).strip()
    status_final = " | ".join(sorted(list(set(alertas_encontrados)))) if alertas_encontrados else "OK"

    return texto_final, status_final

# 3. Execução
print("Processando proteção contra Shadowing...")
resultados = df['Descrição Merchant'].apply(limpar_descricao_robusta)

df['Dicionario Tratada'] = [res[0] for res in resultados]
df['Status Alerta'] = [res[1] for res in resultados]



Selecione o arquivo de catálogo (Excel ou CSV):


Saving DicionarioReferencia.xlsx to DicionarioReferencia.xlsx
Processando proteção contra Shadowing...


In [ ]:
# 4. Exportação
nome_saida = "Catalogo_Revisado.xlsx"
df.to_excel(nome_saida, index=False)
files.download(nome_saida)
print("Sucesso! O arquivo foi processado com segurança total.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Sucesso! O arquivo foi processado com segurança total.
